# 04 - weather data

we fetch hourly weather for every one of the 263 tlc taxi zones from the open-meteo archive api (no api key needed). the idea is to get zone-specific weather by querying at the centroid of each zone polygon, then clean it and validate it.

three steps: fetch, validate, clean.

## fetching weather per zone

we load the tlc zone shapefile, reproject to get accurate centroids, then hit the open-meteo api once per zone. results are cached to disk so re-running skips zones that already finished.

In [ ]:
import io
import os
import time
import zipfile
import geopandas as gpd
import pandas as pd
import requests

CACHE_DIR   = 'cache'
PARQUET_OUT = 'weather_by_zone_2023_2024.parquet'
TLC_ZONES_URL = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip'
ARCHIVE_URL   = 'https://archive-api.open-meteo.com/v1/archive'

WEATHER_VARS = [
    'temperature_2m', 'precipitation', 'snowfall',
    'snow_depth', 'windspeed_10m', 'weathercode',
]

os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
def load_zone_centroids():
    print('downloading nyc taxi zone shapefile...', flush=True)
    r = requests.get(TLC_ZONES_URL, timeout=60)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall('/tmp/taxi_zones')
        names = z.namelist()

    shp_files = [n for n in names if n.endswith('.shp')]
    if not shp_files:
        raise FileNotFoundError(f'no .shp in zip. contents: {names}')
    shp_path = f'/tmp/taxi_zones/{shp_files[0]}'

    gdf = gpd.read_file(shp_path)
    gdf = gdf.to_crs('EPSG:2263')  # projected for accurate centroids
    gdf['centroid'] = gdf.geometry.centroid
    gdf = gdf.to_crs('EPSG:4326')
    gdf['lat'] = gdf['centroid'].to_crs('EPSG:4326').y
    gdf['lon'] = gdf['centroid'].to_crs('EPSG:4326').x

    return gdf[['LocationID', 'zone', 'borough', 'lat', 'lon']].rename(
        columns={'zone': 'zone_name'}
    )

In [ ]:
def fetch_one_zone(lat, lon, location_id, start, end):
    params = {
        'latitude':   f'{lat:.6f}',
        'longitude':  f'{lon:.6f}',
        'start_date': start,
        'end_date':   end,
        'hourly':     ','.join(WEATHER_VARS),
        'timezone':   'America/New_York',
    }

    wait = 10
    for attempt in range(8):
        resp = requests.get(ARCHIVE_URL, params=params, timeout=120)
        if resp.status_code == 429:
            print(f'    429 - waiting {wait}s...', flush=True)
            time.sleep(wait)
            wait = min(wait * 2, 300)
            continue
        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f'zone {location_id}: failed after 8 retries')

    data = resp.json()
    if 'hourly' not in data:
        raise ValueError(f'zone {location_id}: unexpected response')

    df = pd.DataFrame(data['hourly'])
    df['time']       = pd.to_datetime(df['time'])
    df['LocationID'] = location_id
    return df

In [ ]:
def fetch_all_zones(zones, start='2023-01-01', end='2024-12-31'):
    locs  = zones.to_dict('records')
    total = len(locs)

    for i, z in enumerate(locs, 1):
        loc_id     = z['LocationID']
        cache_path = os.path.join(CACHE_DIR, f'zone_{loc_id}.parquet')

        if os.path.exists(cache_path):
            print(f'  [{i:>3}/{total}] zone {loc_id:>3} ({z["zone_name"]}) - cached', flush=True)
            continue

        print(f'  [{i:>3}/{total}] zone {loc_id:>3} ({z["zone_name"]})...', end=' ', flush=True)
        df = fetch_one_zone(z['lat'], z['lon'], loc_id, start, end)
        df.to_parquet(cache_path, index=False)
        print(f'{len(df):,} rows', flush=True)
        time.sleep(6)  # ~10 req/min, stay under free tier

    print('\nassembling from cache...', flush=True)
    frames = []
    for z in locs:
        cache_path = os.path.join(CACHE_DIR, f'zone_{z["LocationID"]}.parquet')
        frames.append(pd.read_parquet(cache_path))

    weather = pd.concat(frames, ignore_index=True)
    weather = weather.merge(
        zones[['LocationID', 'zone_name', 'borough', 'lat', 'lon']],
        on='LocationID', how='left'
    )
    return weather


zones   = load_zone_centroids()
print(f'loaded {len(zones)} zones')
weather = fetch_all_zones(zones, start='2023-01-01', end='2024-12-31')
weather.to_parquet(PARQUET_OUT, index=False)
print(f'saved {len(weather):,} rows')

## validating the output

before cleaning, we check that all 263 zones are present, each has the expected number of hourly records (731 days x 24 hours), and the zone centroids sit inside their own polygon.

In [ ]:
import io
import zipfile
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import Point

PARQUET = 'weather_by_zone_2023_2024.parquet'
TLC_ZONES_URL  = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip'
EXPECTED_ZONES = 263
EXPECTED_HOURS = 731 * 24

df = pd.read_parquet(PARQUET)
print(f'{len(df):,} rows, {df["LocationID"].nunique()} unique zones')

# zone coverage check
zone_counts   = df.groupby('LocationID').size().rename('row_count')
present_zones = set(zone_counts.index)

bad_count = zone_counts[zone_counts != EXPECTED_HOURS]
if bad_count.empty:
    print(f'all {len(present_zones)} zones have exactly {EXPECTED_HOURS} hourly records')
else:
    print(f'warning: {len(bad_count)} zone(s) have unexpected record counts:')
    print(bad_count.to_string())

all_ids = set(range(1, EXPECTED_ZONES + 1))
missing = all_ids - present_zones
extra   = present_zones - all_ids
if missing: print(f'missing zone ids: {sorted(missing)}')
else: print('no missing zones')
if extra: print(f'unexpected zone ids: {sorted(extra)}')

In [ ]:
# centroid-in-polygon check
r = requests.get(TLC_ZONES_URL, timeout=60)
r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall('/tmp/taxi_zones_val')
    shp = [n for n in z.namelist() if n.endswith('.shp')][0]

gdf = gpd.read_file(f'/tmp/taxi_zones_val/{shp}').to_crs('EPSG:4326').set_index('LocationID')

centroids = df[['LocationID', 'lat', 'lon']].drop_duplicates('LocationID').set_index('LocationID')

mismatches = []
for loc_id, row in centroids.iterrows():
    if loc_id not in gdf.index:
        mismatches.append({'LocationID': loc_id, 'issue': 'zone not in shapefile'})
        continue
    point   = Point(row['lon'], row['lat'])
    polygon = gdf.loc[loc_id, 'geometry']
    if not polygon.contains(point) and not polygon.buffer(0.001).contains(point):
        mismatches.append({'LocationID': loc_id, 'zone_name': gdf.loc[loc_id, 'zone'],
                           'issue': 'centroid outside polygon'})

if mismatches:
    print(f'warning: {len(mismatches)} centroid(s) outside their zone polygon')
else:
    print(f'all {len(centroids)} centroids fall within their zone polygon')

# time coverage
df['time'] = pd.to_datetime(df['time'])
t_min, t_max = df['time'].min(), df['time'].max()
print(f'earliest: {t_min}  |  latest: {t_max}')

In [ ]:
# save any issues to csv
issues = []
for loc_id in sorted(missing):
    issues.append({'LocationID': loc_id, 'issue': 'zone missing from parquet'})
for loc_id, count in bad_count.items():
    issues.append({'LocationID': loc_id, 'issue': f'expected {EXPECTED_HOURS} rows, got {count}'})
issues.extend(mismatches)

if issues:
    pd.DataFrame(issues).to_csv('validation_issues.csv', index=False)
    print(f'issue report saved - {len(issues)} issues')
else:
    print('no issues found - data looks complete')

## cleaning the weather data

we check for nulls, flag any physically impossible values (negative snowfall etc.), run 3x iqr outlier detection per variable, then forward-fill any remaining gaps within each zone. the flagged rows are all kept but saved to a report csv.

In [ ]:
import pandas as pd
import numpy as np

PARQUET_IN  = 'weather_by_zone_2023_2024.parquet'
PARQUET_OUT = 'weather_by_zone_2023_2024_clean.parquet'
REPORT_OUT  = 'outlier_report.csv'

NUMERIC_COLS = ['temperature_2m', 'precipitation', 'snowfall', 'snow_depth', 'windspeed_10m']

# hard physical bounds for nyc
PHYSICAL_BOUNDS = {
    'temperature_2m':  (-40,  50),
    'precipitation':   (0,   200),
    'snowfall':        (0,   100),
    'snow_depth':      (0,   200),
    'windspeed_10m':   (0,   150),
}

VALID_WEATHERCODES = set(range(0, 100))

df = pd.read_parquet(PARQUET_IN)
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(['LocationID', 'time']).reset_index(drop=True)
n_original = len(df)
print(f'{n_original:,} rows, {df["LocationID"].nunique()} zones')

In [ ]:
issues = []

# null audit
print('null audit')
null_counts = df[NUMERIC_COLS + ['weathercode']].isnull().sum()
null_pct    = (null_counts / n_original * 100).round(2)
print(pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct}).to_string())

for col in NUMERIC_COLS + ['weathercode']:
    mask = df[col].isnull()
    if mask.any():
        flagged = df[mask][['LocationID', 'time']].copy()
        flagged['column'] = col
        flagged['issue']  = 'null'
        flagged['value']  = np.nan
        issues.append(flagged)

# physical bounds
print('\nphysical bounds check')
for col, (lo, hi) in PHYSICAL_BOUNDS.items():
    mask = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
    n = mask.sum()
    print(f'  {col:20s}: {n:,} rows outside [{lo}, {hi}]')
    if n:
        flagged = df[mask][['LocationID', 'time', col]].copy()
        flagged['column'] = col
        flagged['issue']  = f'outside bounds [{lo},{hi}]'
        flagged.rename(columns={col: 'value'}, inplace=True)
        issues.append(flagged)
        df.loc[df[col] < lo, col] = lo
        df.loc[df[col] > hi, col] = hi

In [ ]:
# 3x iqr outlier detection
print('\niqr outlier detection')
outlier_counts = {}
for col in NUMERIC_COLS:
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    lo  = q1 - 3 * iqr
    hi  = q3 + 3 * iqr
    mask = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
    n = mask.sum()
    outlier_counts[col] = n
    print(f'  {col:20s}: {n:,} outliers  (fence [{lo:.2f}, {hi:.2f}])')
    if n:
        flagged = df[mask][['LocationID', 'time', col]].copy()
        flagged['column'] = col
        flagged['issue']  = f'iqr outlier'
        flagged.rename(columns={col: 'value'}, inplace=True)
        issues.append(flagged)

# fill nulls - forward fill within zone, then backward, then global median as fallback
print('\nfilling nulls')
for col in NUMERIC_COLS + ['weathercode']:
    before = df[col].isnull().sum()
    df[col] = df.groupby('LocationID')[col].transform(lambda s: s.ffill().bfill())
    after = df[col].isnull().sum()
    if before:
        print(f'  {col:20s}: filled {before - after:,} nulls ({after:,} remain)')

remaining = df[NUMERIC_COLS + ['weathercode']].isnull().sum().sum()
if remaining:
    for col in NUMERIC_COLS:
        df[col].fillna(df[col].median(), inplace=True)
    df['weathercode'].fillna(df['weathercode'].mode()[0], inplace=True)
    print(f'  applied global median/mode for {remaining:,} remaining nulls')
else:
    print('  no nulls remain')

df.to_parquet(PARQUET_OUT, index=False)
print(f'saved -> {PARQUET_OUT}')

if issues:
    report = pd.concat(issues, ignore_index=True)
    report = report[['LocationID', 'time', 'column', 'issue', 'value']]
    report.to_csv(REPORT_OUT, index=False)
    print(f'outlier report saved -> {REPORT_OUT}  ({len(report):,} rows)')
else:
    print('no issues found')